# PoliPrompt Demo

Two minimal end-to-end examples:
1. **Text classification** — news topic detection (`examples/BBCNews-tiny/`)
2. **Multimodal classification** — harmful meme detection (`examples/HarmfulMemes-tiny/`)

Both examples run with `testing=True, testing_size=16` for a quick demo.  
A `.env` file containing your API keys is auto-discovered by walking up from the notebook directory.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import yaml

# Locate the repository root by searching upward for pyproject.toml
def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found — are you inside the repository?")

REPO_ROOT = find_repo_root(Path.cwd())
print(f"Repo root: {REPO_ROOT}")

---
## Part 1 — Text Classification (News Topics)

In [ ]:
text_work_station = REPO_ROOT / "examples" / "BBCNews-tiny"
text_config_path  = text_work_station / "config.yaml"
text_prompt_path  = text_work_station / "prompt.txt"

# Write the task description prompt
text_prompt_path.parent.mkdir(parents=True, exist_ok=True)
text_prompt_path.write_text(
    """
    You are a News Reporter. Your task is to classify news text into one of these categories: "politics", "business", "sport", "technology", or "entertainment".

    # INSTRUCTION
    Analyze the text briefly and determine the most appropriate category. 

    # FORMAT REQUIREMENT
    You MUST return your response as a JSON object with exactly two fields:
    1. "label": The selected category name.
    2. "reason": A one-sentence explanation of why this category fits.

    Return ONLY the JSON.
    """
)

text_config = {
    "project": {
        "name": "BBCNews-tiny",
        "modality": "text",
        "work_station": str(text_work_station),
        "data_path": str(REPO_ROOT / "examples" / "BBCNews-tiny" / "BBCNews-tiny.csv"),
        "outfiles_dir": "outfiles",
    },
    "column_mapping": {
        "text_col": "text",
        "image_col": "None",
        "answer_col": "label",
    },
    "user_settings": {
        "options": ["politics", "business", "sport", "technology", "entertainment"],
        "k_shots": 3,
        "lambda_param": 0.5,
        "testing": True,
        "testing_size": 16,
    },
    "models": {
        "embedding_llm": "text-embedding-3-small",
        "primary_llm": "gpt-4o-mini",
        "secondary_llm": "qwen-vl-plus",
        "expert_llm": "o3-mini",
    },
    "retrieval": {"n_exemplars_pool": 16},
    "parallel": {
        "embedding_workers": 4,
        "inference_workers": 2,
        "embedding_batch_size": 5,
    },
    "observability": {"enabled": False},
}

with open(text_config_path, "w") as f:
    yaml.dump(text_config, f, default_flow_style=False, allow_unicode=True)

print(f"Config written to: {text_config_path}")
print(f"Prompt written to: {text_prompt_path}")

In [ ]:
from poliprompt import TextClassifier

text_clf = TextClassifier(
    config_path=text_config_path,
    prompt_path=text_prompt_path,
)

In [ ]:
text_clf.create_few_shot_pool()

In [ ]:
text_clf.optimize_task_description()

In [ ]:
text_clf.annotate()

In [ ]:
text_results = text_clf.evaluate()
print(f"Accuracy : {text_results['accuracy']:.3f}")
print(f"Evaluated: {text_results['n_evaluated']} / {text_results['total']} rows")
print("\nPer-class F1:")
for label, metrics in text_results["per_class"].items():
    print(f"  {label:>15s}  F1={metrics['f1-score']:.3f}  support={metrics['support']}")

---
## Part 2 — Multimodal Classification (Harmful Memes)

In [ ]:
mm_work_station = REPO_ROOT / "examples" / "HarmfulMemes-tiny"
mm_config_path  = mm_work_station / "config.yaml"
mm_prompt_path  = mm_work_station / "prompt.txt"

# Write the task description prompt
mm_prompt_path.parent.mkdir(parents=True, exist_ok=True)
mm_prompt_path.write_text(
    """ 
    You are a Senior Content Moderator. Your task is to classify memes 
    based on the official Hateful Memes Challenge definition.
    
    # CORE DEFINITIONS
    - HATEFUL (label: 1): A direct or indirect attack on people based on
    protected characteristics (race, religion, ethnicity, nationality, 
    immigration status, sex, gender identity, sexual orientation, or disability).
    - NOT HATEFUL (label: 0): Content that lacks a clear attack on a protected group.
    
    # FORMAT REQUIREMENT
    You MUST return your response as a JSON object with exactly two fields:
    1. "label": "0" for not hateful, "1" for hateful.
    2. "reason": A one-sentence explanation.
    Return ONLY the JSON.
    """
)

mm_config = {
    "project": {
        "name": "HarmfulMemes",
        "modality": "multimodal",
        "work_station": str(mm_work_station),
        "data_path": "train-tiny.jsonl",
        "image_dir": ".",
        "outfiles_dir": "outfiles",
    },
    "column_mapping": {
        "text_col": "text",
        "image_col": "img",
        "answer_col": "label",
    },
    "user_settings": {
        "options": ["0", "1"],
        "k_shots": 3,
        "lambda_param": 0.5,
        "testing": True,
        "testing_size": 16,
    },
    "models": {
        "embedding_llm": "qwen3-vl-embedding",
        "primary_llm": "gpt-4o-mini",
        "secondary_llm": "qwen-vl-plus",
        "expert_llm": "o4-mini",
    },
    "retrieval": {"n_exemplars_pool": 16},
    "parallel": {
        "embedding_workers": 4,
        "inference_workers": 2,
        "embedding_batch_size": 5,
    },
    "observability": {"enabled": False},
}

with open(mm_config_path, "w") as f:
    yaml.dump(mm_config, f, default_flow_style=False, allow_unicode=True)

print(f"Config written to: {mm_config_path}")
print(f"Prompt written to: {mm_prompt_path}")

In [ ]:
from poliprompt import MultiModalClassifier

mm_clf = MultiModalClassifier(
    config_path=mm_config_path,
    prompt_path=mm_prompt_path,
)

In [ ]:
mm_clf.create_few_shot_pool()

In [ ]:
mm_clf.optimize_task_description()

In [ ]:
mm_clf.annotate()

In [ ]:
mm_results = mm_clf.evaluate()
print(f"Accuracy : {mm_results['accuracy']:.3f}")
print(f"Evaluated: {mm_results['n_evaluated']} / {mm_results['total']} rows")
print("\nPer-class F1:")
for label, metrics in mm_results["per_class"].items():
    print(f"  {label:>5s}  F1={metrics['f1-score']:.3f}  support={metrics['support']}")